In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/600_test_set.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/5000_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_600_train_5000_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_600_train_5000_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_5000_train_600_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id     subreddit                                      title  \
 0   kg3jun       assault  my assault ruins all of my relationships.   
 1   77d66o         metoo                                     Me Too   
 2  1lmsep9  mentalhealth            why is my attachment like this?   
 3  1nlkz6v   miscarriage                        Lost my twin girls.   
 4  1oa1d0d   miscarriage                       Another miscarriage?   
 
                                             selftext          created_utc  \
 0  this is my first reddit post and i'm bad at ex...   2020-12-19 7:32:02   
 1  After seeing all this hype over the #metoo thi...   2017-10-19 8:42:43   
 2  uhmm hi!!! i don’t really post on reddit or an...  2025-06-28 17:29:31   
 3  We had our anatomy scan two weeks ago, they fo...  2025-09-20T01:11:28   
 4  Summing this up as short as I can. Diagnosed w...  2025-10-18T17:12:00   
 
                                                  url  Tags    __dataset  \
 0  https://www.redd

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test14_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test14_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test14_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test14_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [3]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [4]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test14_emb_A, test14_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_5000_train_600_test.json
